In [ ]:
!pip install spectral einops

import os, time, zipfile
import numpy as np
import scipy.io as sio
import tensorflow as tf
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, cohen_kappa_score, confusion_matrix
from sklearn.metrics import precision_recall_fscore_support
from tensorflow.keras.utils import to_categorical
from einops.layers.tensorflow import Rearrange
import spectral


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 249.0/249.0 kB 6.4 MB/s eta 0:00:00


In [ ]:
# ================= CONFIG =================
dataset = 'SA'           # IP, PU, KSC, SA, Botswana, Houston
windowSize = 25
PCA_BANDS = 15
epochs = 100
batch_size = 128
results_folder = "results_vit"
os.makedirs(results_folder, exist_ok=True)


def loadData(name):
    base = "/content/drive/MyDrive/Colab Notebooks/dataset/"
    if name == 'SA':
        data = sio.loadmat(base+'Salinas_corrected.mat')['salinas_corrected']
        labels = sio.loadmat(base+'Salinas_gt.mat')['salinas_gt']
    return data, labels



In [ ]:

###### PCA + Patch Extraction  #####################
def applyPCA(X, numComponents):
    reshaped = X.reshape(-1, X.shape[2])
    pca = PCA(n_components=numComponents, whiten=True)
    reduced = pca.fit_transform(reshaped)
    return reduced.reshape(X.shape[0], X.shape[1], numComponents)

def padWithZeros(X, margin):
    return np.pad(X, ((margin, margin),(margin, margin),(0,0)), mode='constant')

def createImageCubes(X, y, windowSize):
    margin = windowSize // 2
    Xpad = padWithZeros(X, margin)
    patches, labels = [], []
    for i in range(margin, Xpad.shape[0]-margin):
        for j in range(margin, Xpad.shape[1]-margin):
            if y[i-margin, j-margin] > 0:
                patch = Xpad[i-margin:i+margin+1, j-margin:j+margin+1]
                patches.append(patch)
                labels.append(y[i-margin, j-margin]-1)
    return np.array(patches), np.array(labels)


In [ ]:
###########  Train/Test Split (Count-wise & Percentage-wise)   ###############

def splitTrainTestSet(X, y, train_ratio=None, samples_per_class=None, seed=42):
    np.random.seed(seed)
    train_idx, test_idx = [], []

    for c in np.unique(y):
        idx = np.where(y == c)[0]
        np.random.shuffle(idx)

        if samples_per_class:
            n = min(samples_per_class, len(idx)-1)
        else:
            n = max(1, int(len(idx)*train_ratio))

        train_idx.extend(idx[:n])
        test_idx.extend(idx[n:])

    return X[train_idx], X[test_idx], y[train_idx], y[test_idx]
###########   Vision Transformer Model   ###################

def build_vit(
    image_size=25,
    patch_size=5,
    num_classes=16,
    dim=64,
    depth=6,
    heads=4,
    mlp_dim=128,
    channels=15
):
    num_patches = (image_size // patch_size) ** 2
    patch_dim = patch_size * patch_size * channels

    inputs = tf.keras.layers.Input(shape=(image_size, image_size, channels))

    x = Rearrange(
        'b (h p1) (w p2) c -> b (h w) (p1 p2 c)',
        p1=patch_size, p2=patch_size
    )(inputs)

    x = tf.keras.layers.Dense(dim)(x)

    cls_token = tf.Variable(tf.zeros((1,1,dim)))
    x = tf.concat([tf.repeat(cls_token, tf.shape(x)[0], axis=0), x], axis=1)

    pos_embed = tf.keras.layers.Embedding(num_patches+1, dim)
    positions = tf.range(start=0, limit=num_patches+1)
    x += pos_embed(positions)

    for _ in range(depth):
        x1 = tf.keras.layers.LayerNormalization()(x)
        attn = tf.keras.layers.MultiHeadAttention(
            num_heads=heads, key_dim=dim
        )(x1, x1)
        x = x + attn

        x2 = tf.keras.layers.LayerNormalization()(x)
        mlp = tf.keras.Sequential([
            tf.keras.layers.Dense(mlp_dim, activation='gelu'),
            tf.keras.layers.Dense(dim)
        ])(x2)
        x = x + mlp

    x = tf.keras.layers.LayerNormalization()(x)
    x = x[:, 0]
    outputs = tf.keras.layers.Dense(num_classes, activation='softmax')(x)

    model = tf.keras.Model(inputs, outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model


In [ ]:
########### Training & Evaluation   ###################

import tensorflow as tf
from tensorflow.keras.layers import Layer
from tensorflow.keras.initializers import Zeros
from einops.layers.tensorflow import Rearrange

# Redefine the custom layer for adding and repeating the class token
class AddClassTokenLayer(Layer):
    def __init__(self, dim, **kwargs):
        super().__init__(**kwargs)
        self.dim = dim
        self.cls_token = self.add_weight(
            shape=(1, 1, dim),
            initializer=Zeros(),
            trainable=True,
            name='cls_token_weight'
        )

    def call(self, inputs):
        batch_size = tf.shape(inputs)[0]
        repeated_cls_token = tf.repeat(self.cls_token, batch_size, axis=0)
        return tf.concat([repeated_cls_token, inputs], axis=1)

# Redefine the build_vit function with the fix
def build_vit(
    image_size=25,
    patch_size=5,
    num_classes=16,
    dim=64,
    depth=6,
    heads=4,
    mlp_dim=128,
    channels=15
):
    num_patches = (image_size // patch_size) ** 2

    inputs = tf.keras.layers.Input(shape=(image_size, image_size, channels))

    x = Rearrange(
        'b (h p1) (w p2) c -> b (h w) (p1 p2 c)',
        p1=patch_size, p2=patch_size
    )(inputs)

    x = tf.keras.layers.Dense(dim)(x)

    # Integrate the custom layer
    x = AddClassTokenLayer(dim=dim)(x)

    pos_embed = tf.keras.layers.Embedding(num_patches+1, dim)
    positions = tf.range(start=0, limit=num_patches+1)
    x += pos_embed(positions)

    for _ in range(depth):
        x1 = tf.keras.layers.LayerNormalization()(x)
        attn = tf.keras.layers.MultiHeadAttention(
            num_heads=heads, key_dim=dim
        )(x1, x1)
        x = x + attn

        x2 = tf.keras.layers.LayerNormalization()(x)
        mlp = tf.keras.Sequential([
            tf.keras.layers.Dense(mlp_dim, activation='gelu'),
            tf.keras.layers.Dense(dim)
        ])(x2)
        x = x + mlp

    x = tf.keras.layers.LayerNormalization()(x)
    x = x[:, 0]
    outputs = tf.keras.layers.Dense(num_classes, activation='softmax')(x)

    model = tf.keras.Model(inputs, outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

# The rest of the original cell content follows
X, y = loadData(dataset)
X = applyPCA(X, PCA_BANDS)
X, y = createImageCubes(X, y, windowSize)

# Example: 5% training
Xtr, Xte, ytr, yte = splitTrainTestSet(X, y, train_ratio=0.05)

ytr_cat = to_categorical(ytr)
yte_cat = to_categorical(yte)

model = build_vit(
    image_size=windowSize,
    patch_size=5,
    num_classes=ytr_cat.shape[1],
    channels=PCA_BANDS
)

tic = time.time()
model.fit(Xtr, ytr_cat,
          epochs=epochs,
          batch_size=batch_size,
          validation_data=(Xte, yte_cat),
          verbose=2)
train_time = time.time() - tic

pred = np.argmax(model.predict(Xte), axis=1)
oa = accuracy_score(yte, pred)
kappa = cohen_kappa_score(yte, pred)

Epoch 1/100
4/4 - 38s - 9s/step - accuracy: 0.3743 - loss: 2.3531 - val_accuracy: 0.6081 - val_loss: 1.3890
Epoch 2/100
4/4 - 1s - 271ms/step - accuracy: 0.7069 - loss: 1.0844 - val_accuracy: 0.7042 - val_loss: 0.9800
Epoch 3/100
4/4 - 1s - 212ms/step - accuracy: 0.7980 - loss: 0.7317 - val_accuracy: 0.7652 - val_loss: 0.7747
Epoch 4/100
4/4 - 1s - 214ms/step - accuracy: 0.8475 - loss: 0.5119 - val_accuracy: 0.7920 - val_loss: 0.6423
Epoch 5/100
4/4 - 1s - 215ms/step - accuracy: 0.8970 - loss: 0.3647 - val_accuracy: 0.8239 - val_loss: 0.5707
Epoch 6/100
4/4 - 1s - 352ms/step - accuracy: 0.9347 - loss: 0.2677 - val_accuracy: 0.8242 - val_loss: 0.5511
Epoch 7/100
4/4 - 1s - 199ms/step - accuracy: 0.9465 - loss: 0.1975 - val_accuracy: 0.8377 - val_loss: 0.5169
Epoch 8/100
4/4 - 1s - 202ms/step - accuracy: 0.9723 - loss: 0.1407 - val_accuracy: 0.8402 - val_loss: 0.5131
Epoch 9/100
4/4 - 1s - 205ms/step - accuracy: 0.9842 - loss: 0.0923 - val_accuracy: 0.8308 - val_loss: 0.5342
Epoch 10/100

In [2]:
# ============================================================
# INSTALL DEPENDENCIES
# ============================================================
!pip install -q spectral einops

# ============================================================
# IMPORTS
# ============================================================
import os, time, zipfile
import numpy as np
import matplotlib.pyplot as plt
import scipy.io as sio
import tensorflow as tf
from sklearn.decomposition import PCA
from sklearn.metrics import (
    confusion_matrix, accuracy_score,
    classification_report, cohen_kappa_score,
    precision_recall_fscore_support
)
from tensorflow.keras.utils import to_categorical
from einops.layers.tensorflow import Rearrange
import spectral
from tensorflow.keras.layers import Layer
from tensorflow.keras.initializers import Zeros

# ============================================================
# CONFIGURATION
# ============================================================
dataset = 'Ho'                     # IP, SA, PU, KSC, Bo, Ho
windowSize = 25
PCA_BANDS = 15
epochs = 100
batch_size = 128
train_ratio = None                 # percentage-wise split
samples_per_class = 5           # set integer for count-wise split

model_name = "Ho_PCA15_ViT_5_spc"
results_folder = "results_vit"
os.makedirs(results_folder, exist_ok=True)

# ============================================================
# DATA LOADING
# ============================================================
def loadData(name):
    base = "/content/drive/MyDrive/Colab Notebooks/dataset/"
    if name == 'IP':
        data = sio.loadmat(base+'Indian_pines_corrected.mat')['indian_pines_corrected']
        labels = sio.loadmat(base+'Indian_pines_gt.mat')['indian_pines_gt']
    elif name == 'SA':
        data = sio.loadmat(os.path.join(base, 'Salinas_corrected.mat'))['salinas_corrected']
        labels = sio.loadmat(os.path.join(base, 'Salinas_gt.mat'))['salinas_gt']
    elif name == 'Ho':
        data = sio.loadmat(os.path.join(base, 'Houston.mat'))['houston']
        labels = sio.loadmat(os.path.join(base, 'Houston_gt.mat'))['houston_gt']
    elif name == 'PU':
        data = sio.loadmat(os.path.join(base, 'PaviaU.mat'))['paviaU']
        labels = sio.loadmat(os.path.join(base, 'PaviaU_gt.mat'))['paviaU_gt']
    elif name == 'Bo':
        data = sio.loadmat(os.path.join(base, 'Botswana.mat'))['Botswana']
        labels = sio.loadmat(os.path.join(base, 'Botswana_gt.mat'))['Botswana_gt']
    elif name == 'KSC':
        data = sio.loadmat(os.path.join(base, 'KSC.mat'))['KSC']
        labels = sio.loadmat(os.path.join(base, 'KSC_gt.mat'))['KSC_gt']
    else:
        raise ValueError("Dataset not configured")
    return data, labels

# ============================================================
# PREPROCESSING
# ============================================================
def applyPCA(X, n_components):
    Xr = X.reshape(-1, X.shape[2])
    pca = PCA(n_components=n_components, whiten=True)
    Xp = pca.fit_transform(Xr)
    return Xp.reshape(X.shape[0], X.shape[1], n_components)

def padWithZeros(X, margin):
    return np.pad(X, ((margin, margin),(margin, margin),(0,0)), mode='constant')

def createImageCubes(X, y, windowSize):
    margin = windowSize // 2
    Xp = padWithZeros(X, margin)
    patches, labels = [], []
    for r in range(margin, Xp.shape[0]-margin):
        for c in range(margin, Xp.shape[1]-margin):
            if y[r-margin, c-margin] > 0:
                patches.append(Xp[r-margin:r+margin+1, c-margin:c+margin+1])
                labels.append(y[r-margin, c-margin]-1)
    return np.array(patches), np.array(labels)

# ============================================================
# TRAIN / TEST SPLIT
# ============================================================
def splitTrainTestSet(X, y, train_ratio=None, samples_per_class=None):
    np.random.seed(42)
    tr, te = [], []
    for c in np.unique(y):
        idx = np.where(y==c)[0]
        np.random.shuffle(idx)
        n = min(samples_per_class, len(idx)-1) if samples_per_class else max(1,int(len(idx)*train_ratio))
        tr.extend(idx[:n]); te.extend(idx[n:])
    return X[tr], X[te], y[tr], y[te]

# ============================================================
# LOAD AND PREPARE DATA
# ============================================================
X, y = loadData(dataset)
X = applyPCA(X, PCA_BANDS)
X, y = createImageCubes(X, y, windowSize)

Xtr, Xte, ytr, yte = splitTrainTestSet(
    X, y, train_ratio=train_ratio, samples_per_class=samples_per_class
)

ytr_cat = to_categorical(ytr)
yte_cat = to_categorical(yte)

# Custom layer to add and repeat the class token
class AddClassTokenLayer(Layer):
    def __init__(self, dim, **kwargs):
        super().__init__(**kwargs)
        self.dim = dim
        self.cls_token = self.add_weight(
            shape=(1, 1, dim),
            initializer=Zeros(),
            trainable=True,
            name='cls_token_weight'
        )

    def call(self, inputs):
        batch_size = tf.shape(inputs)[0]
        repeated_cls_token = tf.repeat(self.cls_token, batch_size, axis=0)
        return tf.concat([repeated_cls_token, inputs], axis=1)

# ============================================================
# VISION TRANSFORMER MODEL
# ============================================================
def build_vit(image_size=25, patch_size=5, channels=30,
              num_classes=16, dim=64, depth=6, heads=4, mlp_dim=128):

    num_patches = (image_size // patch_size) ** 2

    inputs = tf.keras.Input(shape=(image_size,image_size,channels))

    x = Rearrange(
        'b (h p1) (w p2) c -> b (h w) (p1 p2 c)',
        p1=patch_size, p2=patch_size
    )(inputs)

    x = tf.keras.layers.Dense(dim)(x)

    # Use the custom layer for the class token
    x = AddClassTokenLayer(dim=dim)(x)

    pos_embed = tf.keras.layers.Embedding(num_patches+1, dim)
    x = x + pos_embed(tf.range(num_patches+1))

    for _ in range(depth):
        x1 = tf.keras.layers.LayerNormalization()(x)
        attn = tf.keras.layers.MultiHeadAttention(heads, dim)(x1,x1)
        x = x + attn
        x2 = tf.keras.layers.LayerNormalization()(x)
        mlp = tf.keras.Sequential([
            tf.keras.layers.Dense(mlp_dim, activation='gelu'),
            tf.keras.layers.Dense(dim)
        ])(x2)
        x = x + mlp

    x = tf.keras.layers.LayerNormalization()(x)
    x = x[:,0]
    outputs = tf.keras.layers.Dense(num_classes, activation='softmax')(x)

    model = tf.keras.Model(inputs, outputs)
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

model = build_vit(
    image_size=windowSize,
    patch_size=5,
    channels=PCA_BANDS,
    num_classes=ytr_cat.shape[1]
)

model.summary()

# ============================================================
# TRAINING
# ============================================================
tic = time.perf_counter()
history = model.fit(
    Xtr, ytr_cat,
    epochs=epochs,
    batch_size=batch_size,
    validation_data=(Xte, yte_cat),
    verbose=2
)
train_time = time.perf_counter() - tic

# ============================================================
# TRAINING CURVES
# ============================================================
plt.figure()
plt.plot(history.history['accuracy'], label='Train Acc')
plt.plot(history.history['val_accuracy'], label='Val Acc')
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.xlabel("Epochs"); plt.ylabel("Value"); plt.legend()
plt.savefig(os.path.join(results_folder, model_name+"_training.png"))
plt.close()

# ============================================================
# EVALUATION
# ============================================================
tic = time.perf_counter()
pred = np.argmax(model.predict(Xte), axis=1)
test_time = time.perf_counter() - tic

oa = accuracy_score(yte, pred)
cm = confusion_matrix(yte, pred)
each_acc = np.nan_to_num(np.diag(cm)/cm.sum(axis=1))
aa = np.mean(each_acc)
kappa = cohen_kappa_score(yte, pred)

report = classification_report(yte, pred, digits=4)

# ============================================================
# SAVE RESULTS
# ============================================================
with open(os.path.join(results_folder, model_name+"_results.txt"), "w") as f:
    f.write(f"Training time: {train_time:.2f}s\n")
    f.write(f"Testing time: {test_time:.2f}s\n")
    f.write(f"OA: {oa*100:.2f}%\n")
    f.write(f"AA: {aa*100:.2f}%\n")
    f.write(f"Kappa: {kappa*100:.2f}%\n\n")
    f.write(report)

# ============================================================
# FULL MAP PREDICTION
# ============================================================
X_full, y_full = loadData(dataset)
X_full = applyPCA(X_full, PCA_BANDS)
Xp = padWithZeros(X_full, windowSize//2)

out = np.zeros_like(y_full)
coords, patches = [], []

for i in range(y_full.shape[0]):
    for j in range(y_full.shape[1]):
        if y_full[i,j]>0:
            patches.append(Xp[i:i+windowSize,j:j+windowSize])
            coords.append((i,j))

patches = np.array(patches)
preds = np.argmax(model.predict(patches,batch_size=256), axis=1)

for (i,j),p in zip(coords,preds):
    out[i,j]=p+1

spectral.save_rgb(os.path.join(results_folder, model_name+"_map.jpg"),
                  out.astype(int), colors=spectral.spy_colors)
spectral.save_rgb(os.path.join(results_folder, model_name+"_gt.jpg"),
                  y_full.astype(int), colors=spectral.spy_colors)

# ============================================================
# ZIP OUTPUTS
# ============================================================
zip_path = model_name+"_outputs.zip"
with zipfile.ZipFile(zip_path,'w') as z:
    for f in os.listdir(results_folder):
        z.write(os.path.join(results_folder,f), f)

print("✅ ViT experiment completed successfully.")
print("📦 Outputs zipped at:", zip_path)

Model: "functional_13"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_7       │ (None, 25, 25,    │          0 │ -                 │
│ (InputLayer)        │ 15)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rearrange_1         │ (None, 25, 375)   │          0 │ input_layer_7[0]… │
│ (Rearrange)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_14 (Dense)    │ (None, 25, 64)    │     24,064 │ rearrange_1[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_class_token_la… │ (None, 26, 64)    │         64 │ dense_14[0][0]    │
│ (AddClassTokenLaye… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_13 (Add)        │ (None, 26, 64)    │          0 │ add_class_token_… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 26, 64)    │        128 │ add_13[0][0]      │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 26, 64)    │     66,368 │ layer_normalizat… │
│ (MultiHeadAttentio… │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_14 (Add)        │ (None, 26, 64)    │          0 │ add_13[0][0],     │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 26, 64)    │        128 │ add_14[0][0]      │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sequential_6        │ (None, 26, 64)    │     16,576 │ layer_normalizat… │
│ (Sequential)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_15 (Add)        │ (None, 26, 64)    │          0 │ add_14[0][0],     │
│                     │                   │            │ sequential_6[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 26, 64)    │        128 │ add_15[0][0]      │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 26, 64)    │     66,368 │ layer_normalizat… │
│ (MultiHeadAttentio… │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_16 (Add)        │ (None, 26, 64)    │          0 │ add_15[0][0],     │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 26, 64)    │        128 │ add_16[0][0]      │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sequential_7        │ (None, 26, 64)    │     16,576 │ layer_normalizat… │
│ (Sequential)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_17 (Add)        │ (None, 26, 64)    │          0 │ add_16[0][0],     │
│                     │                   │            │ sequential_7[0][… │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 524,431 (2.00 MB)

 Trainable params: 524,431 (2.00 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
1/1 - 30s - 30s/step - accuracy: 0.0133 - loss: 3.4554 - val_accuracy: 0.3499 - val_loss: 2.1454
Epoch 2/100
1/1 - 1s - 1s/step - accuracy: 0.4400 - loss: 1.6606 - val_accuracy: 0.5069 - val_loss: 1.7445
Epoch 3/100
1/1 - 1s - 871ms/step - accuracy: 0.7733 - loss: 1.0553 - val_accuracy: 0.5356 - val_loss: 1.5870
Epoch 4/100
1/1 - 1s - 875ms/step - accuracy: 0.8533 - loss: 0.7617 - val_accuracy: 0.5765 - val_loss: 1.4879
Epoch 5/100
1/1 - 1s - 877ms/step - accuracy: 0.9600 - loss: 0.5415 - val_accuracy: 0.6175 - val_loss: 1.4291
Epoch 6/100
1/1 - 1s - 859ms/step - accuracy: 0.9467 - loss: 0.3924 - val_accuracy: 0.6508 - val_loss: 1.3952
Epoch 7/100
1/1 - 1s - 1s/step - accuracy: 0.9867 - loss: 0.2916 - val_accuracy: 0.6642 - val_loss: 1.3745
Epoch 8/100
1/1 - 1s - 865ms/step - accuracy: 0.9867 - loss: 0.2131 - val_accuracy: 0.6633 - val_loss: 1.3624
Epoch 9/100
1/1 - 1s - 864ms/step - accuracy: 1.0000 - loss: 0.1591 - val_accuracy: 0.6610 - val_loss: 1.3516
Epoch 10/100
1/1 